# Option A : LSTM Simple - OHLCV Brut

**Approche minimaliste** : Utilise uniquement les données OHLCV brutes sans feature engineering.

**Avantages :**
- Moins d'overfitting
- Le modèle apprend ses propres patterns
- Plus rapide à entraîner

**Runtime recommandé:** GPU T4

In [ ]:
!pip install yfinance pandas numpy scikit-learn tensorflow matplotlib --quiet
print("Dependencies installed!")

In [ ]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import json
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

import tensorflow as tf
from tensorflow import keras
from keras.models import Sequential
from keras.layers import LSTM, Dense, Dropout, BatchNormalization, Bidirectional
from keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from keras.optimizers import Adam

print(f"TensorFlow: {tf.__version__}")
print(f"GPU: {tf.config.list_physical_devices('GPU')}")

In [ ]:
# ============ CONFIGURATION ============
CONFIG = {
    "symbol": "BTC-USD",
    "years_of_data": 4,              # Plus de données
    "sequence_length": 30,           # 30 jours de contexte (moins = moins d'overfitting)
    
    # OHLCV uniquement
    "feature_columns": ["Open", "High", "Low", "Close", "Volume"],
    "target_columns": ["Open", "High", "Low", "Close"],
    
    # Modèle simplifié
    "lstm_units": 64,
    "dropout_rate": 0.3,
    
    # Entraînement
    "batch_size": 32,
    "epochs": 150,
    "learning_rate": 0.001,
    "validation_split": 0.15,
    "test_split": 0.1,
}

print("Configuration Option A (Simple):")
for k, v in CONFIG.items():
    print(f"  {k}: {v}")

In [ ]:
# Télécharger les données
end_date = datetime.now()
start_date = end_date - timedelta(days=CONFIG["years_of_data"] * 365)

print(f"Downloading {CONFIG['symbol']} data...")
df = yf.download(CONFIG["symbol"], start=start_date, end=end_date, progress=False)
df = df.dropna()

print(f"Downloaded {len(df)} days")
print(f"Date range: {df.index[0].strftime('%Y-%m-%d')} to {df.index[-1].strftime('%Y-%m-%d')}")
print(f"Price range: ${df['Low'].min():,.0f} - ${df['High'].max():,.0f}")

# Visualiser
plt.figure(figsize=(14, 5))
plt.plot(df.index, df['Close'], linewidth=1)
plt.title(f'BTC Price History ({CONFIG["years_of_data"]} years)')
plt.xlabel('Date')
plt.ylabel('Price (USD)')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Préparer les données - UNIQUEMENT OHLCV
feature_cols = CONFIG["feature_columns"]
target_cols = CONFIG["target_columns"]
seq_len = CONFIG["sequence_length"]

# Extraire les données
data = df[feature_cols].values
targets = df[target_cols].values

# Normalisation séparée pour features et targets
scaler_features = MinMaxScaler(feature_range=(0, 1))
scaler_targets = MinMaxScaler(feature_range=(0, 1))

data_scaled = scaler_features.fit_transform(data)
targets_scaled = scaler_targets.fit_transform(targets)

# Créer les séquences
X, y = [], []
for i in range(seq_len, len(data_scaled)):
    X.append(data_scaled[i-seq_len:i])
    y.append(targets_scaled[i])

X = np.array(X)
y = np.array(y)

# Split temporel
test_size = int(len(X) * CONFIG["test_split"])
val_size = int(len(X) * CONFIG["validation_split"])
train_size = len(X) - test_size - val_size

X_train, y_train = X[:train_size], y[:train_size]
X_val, y_val = X[train_size:train_size+val_size], y[train_size:train_size+val_size]
X_test, y_test = X[train_size+val_size:], y[train_size+val_size:]

print(f"\nData shapes:")
print(f"  X_train: {X_train.shape}")
print(f"  X_val: {X_val.shape}")
print(f"  X_test: {X_test.shape}")
print(f"  Features: {feature_cols}")
print(f"  Targets: {target_cols}")

In [ ]:
# Modèle LSTM Simple avec Bidirectional
model = Sequential([
    # Bidirectional LSTM - capture les patterns dans les deux sens
    Bidirectional(LSTM(CONFIG["lstm_units"], return_sequences=True), 
                  input_shape=(X_train.shape[1], X_train.shape[2])),
    Dropout(CONFIG["dropout_rate"]),
    
    # Deuxième couche LSTM
    LSTM(CONFIG["lstm_units"] // 2),
    Dropout(CONFIG["dropout_rate"]),
    
    # Dense layers
    Dense(32, activation='relu'),
    Dense(len(target_cols))  # Output: OHLC
])

model.compile(
    optimizer=Adam(learning_rate=CONFIG["learning_rate"]),
    loss='huber',
    metrics=['mae']
)

model.summary()

In [ ]:
# Callbacks
callbacks = [
    EarlyStopping(monitor='val_loss', patience=20, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_model_A.keras', monitor='val_loss', save_best_only=True, verbose=0)
]

print("Training Option A (Simple OHLCV)...\n")

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size'],
    callbacks=callbacks,
    verbose=1
)

print("\nTraining complete!")

In [ ]:
# Visualiser l'entraînement
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'], label='Train')
axes[0].plot(history.history['val_loss'], label='Validation')
axes[0].set_title('Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history.history['mae'], label='Train')
axes[1].plot(history.history['val_mae'], label='Validation')
axes[1].set_title('MAE')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Évaluation
model = keras.models.load_model('best_model_A.keras')

y_pred_scaled = model.predict(X_test, verbose=0)
y_pred = scaler_targets.inverse_transform(y_pred_scaled)
y_true = scaler_targets.inverse_transform(y_test)

print("=" * 60)
print("OPTION A - RESULTATS (Simple OHLCV)")
print("=" * 60)

metrics = {}
for i, col in enumerate(target_cols):
    mae = mean_absolute_error(y_true[:, i], y_pred[:, i])
    rmse = np.sqrt(mean_squared_error(y_true[:, i], y_pred[:, i]))
    mape = np.mean(np.abs((y_true[:, i] - y_pred[:, i]) / y_true[:, i])) * 100
    r2 = r2_score(y_true[:, i], y_pred[:, i])
    
    # Précision directionnelle
    true_dir = np.diff(y_true[:, i]) > 0
    pred_dir = np.diff(y_pred[:, i]) > 0
    dir_acc = np.mean(true_dir == pred_dir) * 100
    
    metrics[col] = {'mae': mae, 'rmse': rmse, 'mape': mape, 'r2': r2, 'directional_accuracy': dir_acc}
    
    print(f"\n{col}:")
    print(f"  MAE:  ${mae:,.2f}")
    print(f"  MAPE: {mape:.2f}%")
    print(f"  R2:   {r2:.4f}")
    print(f"  Direction Accuracy: {dir_acc:.1f}%")

In [ ]:
# Visualiser les prédictions
n_show = 60

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for i, col in enumerate(target_cols):
    ax = axes[i]
    ax.plot(range(n_show), y_true[-n_show:, i], label='Actual', linewidth=2)
    ax.plot(range(n_show), y_pred[-n_show:, i], label='Predicted', linewidth=2, alpha=0.8)
    ax.set_title(f'{col} - Dir Acc: {metrics[col]["directional_accuracy"]:.1f}%')
    ax.legend()
    ax.grid(True, alpha=0.3)

plt.suptitle('Option A - Simple OHLCV', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Sauvegarder
model.save('btc_lstm_model_A.keras')

scalers = {
    'scaler_features': scaler_features,
    'scaler_targets': scaler_targets
}
joblib.dump(scalers, 'btc_scaler_A.pkl')

config_save = {
    'model_version': '1.0.0-A-simple',
    'trained_at': datetime.now().isoformat(),
    'option': 'A - Simple OHLCV',
    'sequence_length': CONFIG['sequence_length'],
    'feature_columns': feature_cols,
    'target_columns': target_cols,
    'metrics': metrics,
}

with open('training_config_A.json', 'w') as f:
    json.dump(config_save, f, indent=2, default=str)

print("\nFiles saved:")
print("  - btc_lstm_model_A.keras")
print("  - btc_scaler_A.pkl")
print("  - training_config_A.json")

In [ ]:
# Télécharger
try:
    from google.colab import files
    files.download('btc_lstm_model_A.keras')
    files.download('btc_scaler_A.pkl')
    files.download('training_config_A.json')
except:
    print("Download manually from the file browser")